# Bruck-Mürzzuschlag: Gemeinden and Hauptorte
Plots the Gemeinden of GID_2 = `AUT.6.1_2` with one Hauptort point per Gemeinde. Uses `geodata`, `sf`, and base `plot`.

In [1]:
library(geodata)
library(sf)

# Download GADM Austria level 3 (Gemeinden)
aut3 <- gadm('AUT', level = 3, path = tempdir())
aut3_sf <- st_as_sf(aut3)

cat('CRS:', st_crs(aut3_sf)$input, '\n')
cat('Total Gemeinden in dataset:', nrow(aut3_sf), '\n')

Loading required package: terra

terra 1.9.11

Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.4.0; sf_use_s2() is TRUE



CRS: WGS 84 
Total Gemeinden in dataset: 2100 


In [2]:
# Filter Gemeinden for Bruck-Mürzzuschlag (GID_2 = AUT.6.1_2)
bruck <- aut3_sf[aut3_sf$GID_2 == 'AUT.6.1_2', ]

cat('Gemeinden in Bruck-Mürzzuschlag:', nrow(bruck), '\n')
print(bruck[, c('NAME_3', 'GID_3')])

Gemeinden in Bruck-Mürzzuschlag: 19 
Simple feature collection with 19 features and 2 fields
Geometry type: POLYGON
Dimension:     XY
Bounding box:  xmin: 14.93245 ymin: 47.30819 xmax: 15.86343 ymax: 47.82838
Geodetic CRS:  WGS 84
First 10 features:
                       NAME_3        GID_3                       geometry
1438                   Aflenz  AUT.6.1.1_2 POLYGON ((15.26373 47.53491...
1439 Breitenau am Hochlantsch  AUT.6.1.2_2 POLYGON ((15.53532 47.38224...
1440         Bruck an der Mur  AUT.6.1.3_2 POLYGON ((15.24969 47.36313...
1441               Kapfenberg  AUT.6.1.4_2 POLYGON ((15.28109 47.41973...
1442                 Kindberg  AUT.6.1.5_2 POLYGON ((15.41651 47.42481...
1443                Krieglach  AUT.6.1.6_2 POLYGON ((15.64546 47.48358...
1444               Langenwang  AUT.6.1.7_2 POLYGON ((15.69356 47.51978...
1445                Mariazell  AUT.6.1.8_2 POLYGON ((15.09038 47.7421,...
1446             Mürzzuschlag  AUT.6.1.9_2 POLYGON ((15.73871 47.54974...
1447      

In [3]:
# Load Hauptorte shapefile from zip
zip_path <- file.path(getwd(), 'STATISTIK_AUSTRIA_ORT_MP_20250101.zip')
shp_path <- paste0('/vsizip/', zip_path, '/STATISTIK_AUSTRIA_ORT_MP_20250101.shp')
hauptorte <- st_read(shp_path, quiet = TRUE)

cat('CRS:', st_crs(hauptorte)$input, '\n')
cat('Columns:', paste(names(hauptorte), collapse = ', '), '\n')
cat('Total Ortschaften in dataset:', nrow(hauptorte), '\n')
print(head(hauptorte, 3))

CRS: MGI / Austria Lambert 
Columns: gid, g_id, g_name, geometry 
Total Ortschaften in dataset: 16984 
Simple feature collection with 3 features and 3 fields
Geometry type: POINT
Dimension:     XY
Bounding box:  xmin: 640427.1 ymin: 444217.1 xmax: 653719.4 ymax: 455236
Projected CRS: MGI / Austria Lambert
    gid  g_id                         g_name                  geometry
1 00003 00003 Sankt Georgen am Leithagebirge POINT (640427.1 444217.1)
2 00005 00005 Breitenbrunn am Neusiedler See   POINT (653719.4 455236)
3 00006 00006                 Donnerskirchen   POINT (647471.5 449100)


In [4]:
# Reproject to WGS 84 to match GADM, then clip to district
hauptorte_wgs   <- st_transform(hauptorte, st_crs(bruck))
hauptorte_bruck <- st_filter(hauptorte_wgs, bruck)
cat('All Ortschaften in district:', nrow(hauptorte_bruck), '\n')

# Spatial join: tag each Ortschaft with its Gemeinde
ho_joined <- st_join(hauptorte_bruck, bruck[, c('GID_3', 'NAME_3')])

# Fix Latin-1 encoding in names (shapefile uses ISO-8859-1, GADM uses UTF-8)
ho_joined$g_name_utf8 <- iconv(ho_joined$g_name, from = 'latin1', to = 'UTF-8')

# One representative Hauptort per Gemeinde (priority: exact name > prefix > centroid-nearest)
bruck_cents    <- st_centroid(st_geometry(bruck))
hauptorte_main <- do.call(rbind, lapply(seq_len(nrow(bruck)), function(i) {
  pts <- ho_joined[!is.na(ho_joined$GID_3) & ho_joined$GID_3 == bruck$GID_3[i], ]
  if (nrow(pts) == 0L) return(NULL)
  nm <- bruck$NAME_3[i]
  # 1. exact name match
  m <- pts[pts$g_name_utf8 == nm, ]
  if (nrow(m) > 0) return(m[1, ])
  # 2. Gemeinde name is prefix of Ortschaft (e.g. "Aflenz" -> "Aflenz Kurort")
  m <- pts[startsWith(pts$g_name_utf8, nm), ]
  if (nrow(m) > 0) return(m[1, ])
  # 3. Ortschaft name is prefix of Gemeinde (e.g. "Pernegg" -> "Pernegg an der Mur")
  m <- pts[startsWith(nm, pts$g_name_utf8), ]
  if (nrow(m) > 0) return(m[which.min(nchar(m$g_name_utf8)), ])
  # 4. fallback: closest Ortschaft to the Gemeinde centroid
  pts[which.min(st_distance(pts, bruck_cents[i])), ]
}))

cat('One Hauptort per Gemeinde:', nrow(hauptorte_main), '\n')
print(hauptorte_main[, c('g_name_utf8', 'NAME_3')])

All Ortschaften in district: 179 
One Hauptort per Gemeinde: 19 
Simple feature collection with 19 features and 2 fields
Geometry type: POINT
Dimension:     XY
Bounding box:  xmin: 15.07644 ymin: 47.36023 xmax: 15.74566 ymax: 47.77201
Geodetic CRS:  WGS 84
First 10 features:
         g_name_utf8                   NAME_3                  geometry
143    Aflenz Kurort                   Aflenz POINT (15.23667 47.54368)
6       Sankt Erhard Breitenau am Hochlantsch POINT (15.46998 47.38535)
9   Bruck an der Mur         Bruck an der Mur POINT (15.26114 47.41011)
31        Kapfenberg               Kapfenberg POINT (15.28731 47.44172)
95          Kindberg                 Kindberg POINT (15.43168 47.50334)
168        Krieglach                Krieglach  POINT (15.5603 47.54513)
104       Langenwang               Langenwang POINT (15.61918 47.56733)
37         Mariazell                Mariazell  POINT (15.3165 47.77201)
122     Mürzzuschlag             Mürzzuschlag POINT (15.67145 47.60559)
175 

In [ ]:
# Plot: Gemeinden with one Hauptort point + name per Gemeinde
plot(
  st_geometry(bruck),
  col    = '#d9e8f5',
  border = '#555555',
  lwd    = 0.8,
  main   = 'Bruck-Mürzzuschlag: Gemeinden & Hauptorte'
)

plot(
  st_geometry(hauptorte_main),
  pch  = 21,
  bg   = '#e63946',
  col  = '#ffffff',
  cex  = 1.2,
  lwd  = 0.6,
  add  = TRUE
)

coords <- st_coordinates(hauptorte_main)
text(
  x      = coords[, 1],
  y      = coords[, 2],
  labels = hauptorte_main$g_name_utf8,
  cex    = 0.55,
  pos    = 3,      # above the point
  offset = 0.4
)